# 12 • Comunicação e entrega profissional

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [ ]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [ ]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

diario = v.groupby(v['InvoiceDate'].dt.normalize())['valor'].sum()
diario = diario.reindex(pd.date_range(diario.index.min(),diario.index.max(),freq='D'), fill_value=0)
print(diario.head())
# Hipótese: dias não registrados dentro da cobertura tiveram zero venda elegível.

identificados = v.dropna(subset=['CustomerID']).copy()
referencia = v['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = identificados.groupby('CustomerID').agg(
    ultima_compra=('InvoiceDate','max'), frequencia=('InvoiceNo','nunique'), monetario=('valor','sum'))
rfm['recencia_dias'] = (referencia-rfm['ultima_compra'].dt.normalize()).dt.days
print(rfm.head())

antes = p['species'].memory_usage(deep=True)
categorica = p['species'].astype('category')
print('Memória original:',antes,'bytes; categórica:',categorica.memory_usage(deep=True))

def preparar_vendas(bruto):
    obrigatorias = {'InvoiceNo','InvoiceDate','Quantity','UnitPrice','CustomerID','Country'}
    faltantes = obrigatorias - set(bruto.columns)
    if faltantes:
        raise ValueError(f'Colunas ausentes: {sorted(faltantes)}')
    t = bruto.copy()
    t['InvoiceDate'] = pd.to_datetime(t['InvoiceDate'], errors='coerce')
    cancelamento = t['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
    ok = t['Quantity'].gt(0) & t['UnitPrice'].gt(0) & t['InvoiceDate'].notna() & ~cancelamento
    saida = t.loc[ok].copy()
    saida['valor'] = saida['Quantity']*saida['UnitPrice']
    return saida

v2 = preparar_vendas(r)
pd.testing.assert_frame_equal(v, v2)
print('Transformação reproduzida')

### Transformando um resultado em uma conclusão defensável

Escreva uma frase seguindo quatro componentes: medida, população, período e ressalva. Por exemplo: “O país X concentra Y% da receita bruta de vendas positivas elegíveis no período observado; cancelamentos não foram reconciliados”. Calcule X e Y antes de preencher a frase. Não conclua que um país é mais lucrativo: a base não inclui todos os custos necessários para calcular lucro.

No gráfico, plot.barh pede barras horizontais; o índice fornece nomes de países e os valores fornecem comprimentos. set_xlabel explica a medida e a moeda. tight_layout ajusta margens para os textos caberem; savefig grava o arquivo. Fechar a figura com close evita manter várias figuras abertas durante execuções repetidas.

Antes de publicar, peça que outra pessoa consiga responder: de onde vieram os dados, o que cada linha representa, quais registros ficaram de fora e como os totais foram conferidos? Se ela precisa adivinhar essas quatro coisas, o relatório ainda precisa de contexto. Use os projetos finais para praticar esse critério de entrega.

### Aprofundamento e cuidados

Uma análise útil responde a uma decisão concreta. Comece por pergunta, população, período e unidade de análise. Mostre o resultado com a evidência que o sustenta. Separe observação (o que foi medido), interpretação (o que pode explicar) e recomendação (o que fazer ou investigar). Não invente causas a partir de uma tabela agregada.

Para uma série temporal, um gráfico de linha costuma ajudar. Para comparar países, barras ordenadas facilitam a leitura. Inclua unidade no eixo e título com escopo. Não corte o eixo de barras para exagerar diferenças. Para pinguins, distribuições e tamanho da amostra ajudam mais do que apenas uma média.

Uma entrega profissional inclui arquivo bruto preservado, transformação reproduzível, relatório de qualidade, indicadores com definições, validações, instruções de execução e limitações. A base comercial deste curso é histórica: os padrões encontrados não representam o mercado atual. Concluir o curso desenvolve prática; experiência profissional também exige revisão, colaboração e aplicações em contextos diferentes.


## Exemplo guiado

In [ ]:
paises = v.groupby('Country')['valor'].sum().sort_values(ascending=False).head(5)
print(paises.rename('receita_bruta_elegivel_GBP'))
print('Período observado:',v['InvoiceDate'].min(),v['InvoiceDate'].max())

## Exercício 12.1
Prepare uma tabela com receita, faturas e ticket médio por país.

In [ ]:
# Escreva sua resposta aqui


## Exercício 12.2
Exporte os indicadores por país e o RFM para arquivos separados.

In [ ]:
# Escreva sua resposta aqui


## Exercício 12.3
Desenhe barras horizontais dos cinco maiores países com eixo em GBP.

In [ ]:
# Escreva sua resposta aqui


## Exercício 12.4
Escreva uma conclusão factual com participação do maior país e uma limitação.

In [ ]:
# Escreva sua resposta aqui


## Exercício 12.5
Crie um checklist de publicação e explique quando impedir a entrega.

In [ ]:
# Escreva sua resposta aqui
